### **TubeNova AI QA Bot**

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

In [ ]:
yt_api = YouTubeTranscriptApi()
video_fetched = yt_api.fetch(video_id="T-D1OfcDW1M")

In [ ]:
video_fetched

In [ ]:
transcript_fetched = yt_api.list(video_id="T-D1OfcDW1M")
transcript_fetched

In [ ]:
for snipp in transcript_fetched:
    print(snipp.fetch)

In [ ]:
for snipp in video_fetched:
    print(snipp.text)

In [ ]:
import re

In [ ]:
video_url = "https://www.youtube.com/watch?v=T-D1OfcDW1M"

pattern = r'https:\/\/www\.youtube\.com\/watch\?v=([a-zA-Z0-9_-]{11})'
match = re.search(pattern, video_url)

print(match.group(1))

In [ ]:
import logging

def get_logger(name:str):
    
    try:
        if not name:
            raise ValueError("logger name is empty or none")
        
        logging.basicConfig(
            level=logging.INFO,
            format="%(asctime)s - %(name)s - %(levelname)s - %(message)s"
        )
        
        logger = logging.getLogger(name)
        return logger
    
    except ValueError as e:
        print(f"Value error: {e}")
        return str({e})
    except Exception as e:
        print(f"Error in logger: {e}")
        return str({e})

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi
import re


logger = get_logger("data-preprocess-logger")

# import video id 
def get_video_id (video_url:str):
    
    try:
        if not video_url:
            raise ValueError("Video url is empty or none")
        
        pattern = r'https:\/\/www\.youtube\.com\/watch\?v=([a-zA-Z0-9_-]{11})'
        match = re.search(pattern, video_url)
        
        return match.group(1) if match else None
    
    except ValueError as e:
        logger.error(f"Value error: {e}")
    
    except Exception as e:
        logger.error(f"Error in get video id: {e}")
        
def transcript_process(video_url):
    
    try:    
        video_id = get_video_id(video_url)
        
        if video_id is None:
            logger.warning("Video id is None")
            return None
        
        yt_video_api = YouTubeTranscriptApi()
        fecthed_video =  yt_video_api.list(video_id)
        
        if not fecthed_video:
            raise ValueError("Video is not fetched")
        
        logger.info("Video is fetched!")
        
        transcript = ""
        
        for t in fecthed_video:
            if t.language_code=='en':
                if t.is_generated:
                    if len(transcript) == 0:
                        transcript = t.fetch()
                    
                else:
                    transcript = t.fetch()
        
        if not transcript:
            logger.warning("Transcript is not fetched")
            #raise ValueError("Transcript is not fetched")
            return None
        
        logger.info("Transcript has fetched")
        
        return transcript 
        
    
    except ValueError as e:
        logger.error(f"Value error: {e}")
        
    except Exception as e:
        logger.error(f"Error in video fetching: {e}")
    
    

 
    
    
    
    

In [ ]:
transcript=transcript_process(video_url)

In [ ]:
def format_transcript(transcript):
    
    try:
        txt = ""
        
        if not transcript:
            raise ValueError("trainscript is empty or none")
        
        for t in transcript:
            txt += f"Text:{t.text} Start: {t.start}\n"
        
        if len(txt) == 0:
            logger.warning("Formatted transcript is empty and return None")
            return None    
        
        return txt
    
    except ValueError as e:
        logger.error(f"Value error: {e}")
        return None
    except Exception as e:
        logger.error(f"Error in fomartting the transcript: {e}")
        return None    
    
    

In [ ]:
txt = format_transcript(transcript)
txt

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

logger = get_logger("data-preprocess-logger")

def txt_chunking(texts, chunk_size=500, chunk_overlap = 50):
    
    try:
        if not texts or not texts.strip():
            raise ValueError ("Texts are empty or none")
        
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )
        logger.info("Text splitter created")
        
        split_texts = text_splitter.split_text(texts)
        if not split_texts:
            raise ValueError("error is split texts")
        logger.info("Texts are splitted sucessfull")
        
        return split_texts
        
    except ValueError as e:
        logger.error(f"Value error: {e}")
        return None
    
    except Exception as e:
        logger.error(f"Error in text splitting: {e}")
        return None

In [ ]:
spli_texts = txt_chunking(txt)
spli_texts

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
HF_TOKEN = os.getenv("HF_TOKEN")


In [ ]:
from huggingface_hub import login

login(HF_TOKEN)

In [ ]:
from langchain_huggingface import HuggingFacePipeline,ChatHuggingFace

kwargs = {"max_new_tokens": 1000 }
    
pipeline = HuggingFacePipeline.from_model_id(
        model_id="mistralai/Mistral-7B-Instruct-v0.2",
        task="text-generation",
        pipeline_kwargs=kwargs,
        model_kwargs={
            "token": HF_TOKEN
        }
        
    )
    
llm = ChatHuggingFace(llm=pipeline,temperature=0.9)

print(llm.invoke("Explain RAG"))